# 00 — Comprendre et préparer les données

Mode utilisé dans ce support : **smoke**, destiné à vérifier la chaîne complète. Les valeurs commentées proviennent du rapport calculé, jamais de scores inventés.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
REPORT = ROOT / "outputs" / "reports" / "results_smoke.json"
if not REPORT.exists():
    raise FileNotFoundError("Exécutez d'abord : .venv\\Scripts\\python scripts\\run_smoke.py --mode smoke")
results = json.loads(REPORT.read_text(encoding="utf-8"))
sns.set_theme(style="whitegrid")

## Question étudiée

Comment passer d'un panel demi-horaire massif à des observations comparables, sans confondre absence, changement d'heure et imputation ?

Une **journée** est ici un vecteur de 48 créneaux d'un foyer. Une **semaine** est un vecteur aligné de 336 créneaux. Un **foyer** est une unité statistique qui possède plusieurs journées/semaines : il ne faut donc jamais répartir les journées d'un même foyer entre apprentissage et test.

La variable est une **puissance** et les horodatages sont en **UTC**, conformément à la confirmation métier. L'unité exacte de puissance reste à renseigner : les axes conservent donc l'« unité source ». Multiplier par 0,5 h ne permettra de calculer une énergie que lorsque l'unité aura été confirmée.

## Du format long à une matrice d'apprentissage

La source contient une ligne par mesure. Pour un auto-encodeur de journées, on veut **une journée d'un foyer par ligne** et les 48 demi-heures en colonnes. La cellule suivante montre explicitement la transposition sur quelques foyers, puis la création d'un `DataLoader`. Sur des années entières, le même geste produit 17 520 colonnes (voir le notebook 07).

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from cer_ae.data import load_half_hourly

ids = pd.read_parquet(ROOT/'data/processed/smoke_selected_clients.parquet', columns=['ID']).ID.head(3).astype(int).tolist()
long = load_half_hourly(ROOT/'data/df_join.parquet', ids, '2009-07-20', '2009-07-23',
                        {'client':'ID', 'timestamp':'time', 'value':'puissance', 'temperature':'temp'})
long['jour'] = long.timestamp.dt.floor('D')
long['creneau'] = long.timestamp.dt.hour*2 + long.timestamp.dt.minute//30
wide = long.pivot(index=['client_id','jour'], columns='creneau', values='value')
assert wide.shape[1] == 48 and wide.notna().all().all()
display(long.head(), wide.iloc[:3,:8])
print('Matrice journées × créneaux :', wide.shape)
loader = DataLoader(TensorDataset(torch.tensor(wide.to_numpy(dtype='float32'))),
                    batch_size=4, shuffle=True)
print('Un mini-lot :', next(iter(loader))[0].shape)

In [ ]:
audit = json.loads((ROOT/'outputs/reports/data_audit.json').read_text(encoding='utf-8'))
pd.DataFrame({k: [v[0] if len(v)==1 else f'{len(v)} lignes'] for k,v in audit.items()}).T

## Couverture et choix de population

Le diagnostic hors mémoire contrôle schémas, clés, correspondance ID/PDL, plages, grille calendaire et cohérence quotidienne. La sélection par défaut retient les foyers complets sur la période de l'expérience, au lieu de prendre l'intersection aveugle de tous les foyers. Sur le jeu réel, 3 951 des 4 091 foyers couvrent toute la période globale (96,6 %).

La grille observée comporte toujours 48 positions par jour UTC complet. UTC n'ayant pas de transition saisonnière, les changements d'heure locaux irlandais ne créent ici ni journée de 46 ni journée de 50 points.

In [ ]:
windows = np.load(ROOT/'data/processed/smoke_daily_windows.npz')
meta = pd.read_parquet(ROOT/'data/processed/smoke_daily_metadata.parquet')
summary = pd.Series({'foyers': meta.client_id.nunique(), 'journées': len(meta),
                     'points observés': int(windows['observed_mask'].sum()),
                     'points imputés': int((windows['status']>0).sum()),
                     'fenêtres quasi-nulles écartées': results['shape_near_zero_excluded']})
display(summary.to_frame('valeur'))
fig, ax = plt.subplots(figsize=(10,3))
ax.plot(windows['values'][0], marker='.', label='observé')
imputed = windows['status'][0] > 0
ax.scatter(np.flatnonzero(imputed), windows['values'][0][imputed], color='tab:red', label='imputé', zorder=3)
ax.set(xlabel='Créneau demi-horaire', ylabel='Valeur (unité source)', title='Exemple de journée et masque')
ax.legend(); plt.show()

## Imputation et normalisations

Pour un trou à $t$, la règle rétrospective est

$$\hat x_t = \frac{x_{t-7j}+x_{t+7j}}{2}.$$

Seules les valeurs originellement observées servent de donneurs ; le code conserve quatre états : observé, imputé avec deux donneurs, imputé avec un donneur, toujours manquant. Une colonne de partition peut interdire tout franchissement apprentissage/validation/test. Cette méthode utilise le futur et n'est **pas** temps réel.

Deux vues sont préparées : (1) standardisation commune ajustée sur l'apprentissage, qui préserve les écarts de niveau ; (2) division par la moyenne de la fenêtre, qui isole la forme en conservant le niveau séparément. Les moyennes proches de zéro sont explicitement exclues de la seconde vue.

In [ ]:
from cer_ae.preprocessing import normalize_shape
shape, level, valid = normalize_shape(windows['values'])
fig, axes = plt.subplots(1,2,figsize=(11,3))
axes[0].plot(windows['values'][:8].T, alpha=.5); axes[0].set_title('Niveaux conservés')
axes[1].plot(shape[:8].T, alpha=.5); axes[1].set_title('Formes / moyenne de fenêtre')
for ax in axes: ax.set_xlabel('Créneau'); ax.set_ylabel('Valeur')
plt.show()

## Limites et questions aux étudiants

- L'absence de trous dans cette version des données empêche d'évaluer empiriquement l'imputation ; elle est vérifiée par des tests synthétiques ciblés.
- Quelle documentation faudrait-il obtenir avant de convertir la variable `puissance` en kWh ?
- Pourquoi une journée de 46 ou 50 points ne devrait-elle pas être traitée comme une panne de mesure ?